In [2]:
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt


@dataclass
class Params:
    mass: float = 0.42
    gravity: float = 9.81
    damping: float = 0.45
    force_k: float = 1.8e-4
    gap_offset: float = 0.004
    gap_min: float = 0.006
    gap_max: float = 0.024
    current_min: float = 0.0
    current_max: float = 4.2
    reference: float = 0.014


class RandomFeaturePINN:
   # """Physics-informed random-feature neural surrogate F(x,i)=i^2 q_theta(x).

    #q uses a tanh hidden layer.  Output weights are obtained from one linear
    #least-squares problem containing both sparse force observations and the
    #governing residual q'(x)+2q(x)/(x+x0)=0.
    #"""#
    def __init__(self, p: Params, n_hidden: int = 36, seed: int = 7):
        self.p = p
        rng = np.random.default_rng(seed)
        self.center = 0.5 * (p.gap_min + p.gap_max)
        self.scale = 0.5 * (p.gap_max - p.gap_min)
        self.a = rng.normal(0.0, 1.4, n_hidden)
        self.b = rng.uniform(-1.0, 1.0, n_hidden)
        self.w = np.zeros(n_hidden + 1)

    def features(self, x):
        z = (np.asarray(x)[..., None] - self.center) / self.scale
        return np.concatenate([np.ones((*z.shape[:-1], 1)), np.tanh(z*self.a+self.b)], axis=-1)

    def dfeatures(self, x):
        z = (np.asarray(x)[..., None] - self.center) / self.scale
        h = np.tanh(z*self.a+self.b)
        dh = (1.0-h*h)*self.a/self.scale
        return np.concatenate([np.zeros((*z.shape[:-1], 1)), dh], axis=-1)

    def train(self, n_data=26, n_physics=180, noise=0.015, seed=11):
        rng = np.random.default_rng(seed)
        x = rng.uniform(self.p.gap_min, self.p.gap_max, n_data)
        i = rng.uniform(0.45, self.p.current_max, n_data)
        f = true_force(x, i, self.p) * (1.0 + noise*rng.normal(size=n_data))
        # Scale each block so data and PDE residuals have comparable influence.
        A_data = (i*i)[:, None] * self.features(x)
        y_data = f
        xp = np.linspace(self.p.gap_min, self.p.gap_max, n_physics)
        A_phys = self.dfeatures(xp) + 2.0*self.features(xp)/(xp[:, None]+self.p.gap_offset)
        # Dimensionless row normalisation preserves each equation's zero set.
        A_data /= np.maximum(np.linalg.norm(A_data, axis=1, keepdims=True), 1e-12)
        y_data /= np.maximum(np.linalg.norm((i*i)[:, None]*self.features(x), axis=1), 1e-12)
        A_phys /= np.maximum(np.linalg.norm(A_phys, axis=1, keepdims=True), 1e-12)
        A = np.vstack([A_data, 2.0*A_phys, 1e-4*np.eye(len(self.w))])
        y = np.concatenate([y_data, np.zeros(n_physics), np.zeros(len(self.w))])
        self.w = np.linalg.lstsq(A, y, rcond=None)[0]

    def q(self, x):
        return np.maximum(self.features(x) @ self.w, 1e-5)

    def force(self, x, i):
        return np.asarray(i)**2 * self.q(x)


def true_force(x, i, p: Params):
    # Includes an unmodelled weak saturation/fringing term.
    ideal = p.force_k*np.asarray(i)**2/(np.asarray(x)+p.gap_offset)**2
    return ideal*(1.0 - 0.045*np.tanh(0.9*(np.asarray(i)-2.2)))


class AdaptiveSynergeticController:
    def __init__(self, p: Params, surrogate: RandomFeaturePINN, adaptive=True, safety=True):
        self.p, self.net = p, surrogate
        self.adaptive, self.safety = adaptive, safety
        self.centers = np.linspace(-0.010, 0.010, 9)
        self.width = 0.004
        self.weights = np.zeros(len(self.centers))
        self.prev_v = 0.0
        self.dhat = 0.0

    def reset(self):
        self.weights[:] = 0.0
        self.prev_v = self.dhat = 0.0

    def __call__(self, x, v, dt):
        p = self.p
        e = x-p.reference
        # Terminal/synergetic macro-variable psi = v + lambda*e.
        lam, tau = 15.0, 0.055
        psi = v + lam*e
        phi = np.exp(-0.5*((e-self.centers)/self.width)**2)
        raw_hat = float(self.weights @ phi) if self.adaptive else 0.0
        self.dhat = float(np.clip(raw_hat, -3.0, 3.0))
        # tau*psi_dot + psi = 0, with robust saturation for residual error.
        a_cmd = -lam*v - psi/tau - 2.0*np.tanh(psi/0.015)

        # Relative-degree-two exponential CBF bounds on acceleration.
        if self.safety:
            k0, k1 = 180.0, 28.0
            a_lo = -k1*v - k0*(x-p.gap_min)
            a_hi = -k1*v + k0*(p.gap_max-x)
            a_cmd = float(np.clip(a_cmd, a_lo, a_hi))

        # m*xddot = mg + d - c*v - Fmag; invert PINN force surrogate.
        desired_force = p.mass*p.gravity + self.dhat - p.damping*v - p.mass*a_cmd
        desired_force = float(np.clip(desired_force, 0.0, self.net.force(x, p.current_max)))
        current = np.sqrt(desired_force/self.net.q(x))
        current = float(np.clip(current, p.current_min, p.current_max))

        # RBF neural adaptation. Projection/leakage prevents estimator drift.
        if self.adaptive:
            gamma, leakage = 32.0, 1.1
            self.weights += dt*(gamma*phi*psi - leakage*self.weights)
            self.weights = np.clip(self.weights, -1.5, 1.5)
        return current, self.dhat, psi


def external_thrust(t):
    #"""Unknown downward axial load [N], never supplied to the controller."""
    step = 1.05 if t >= 1.5 else 0.0
    sinusoid = 0.32*np.sin(2*np.pi*0.72*(t-3.0)) if t >= 3.0 else 0.0
    pulse = -0.75 if 4.8 <= t <= 5.25 else 0.0
    return step + sinusoid + pulse


def simulate(ctrl, p, duration=7.0, dt=0.001):
    ctrl.reset()
    n = int(duration/dt)+1
    out = {k: np.zeros(n) for k in ['t','x','v','i','d','dhat','psi']}
    out['t'] = np.arange(n)*dt
    out['x'][0], out['v'][0] = 0.0185, 0.0

    def dynamics(x, v, current, d):
        return v, (p.mass*p.gravity+d-p.damping*v-true_force(x, current, p))/p.mass

    for k in range(n-1):
        t, x, v = out['t'][k], out['x'][k], out['v'][k]
        current, dhat, psi = ctrl(x, v, dt)
        d = external_thrust(t)
        # RK4 with zero-order-held current and disturbance over one sample.
        k1 = dynamics(x, v, current, d)
        k2 = dynamics(x+dt*k1[0]/2, v+dt*k1[1]/2, current, d)
        k3 = dynamics(x+dt*k2[0]/2, v+dt*k2[1]/2, current, d)
        k4 = dynamics(x+dt*k3[0], v+dt*k3[1], current, d)
        out['x'][k+1] = x+dt*(k1[0]+2*k2[0]+2*k3[0]+k4[0])/6
        out['v'][k+1] = v+dt*(k1[1]+2*k2[1]+2*k3[1]+k4[1])/6
        out['i'][k], out['d'][k], out['dhat'][k], out['psi'][k] = current, d, dhat, psi
    for key in ['i','d','dhat','psi']:
        out[key][-1] = out[key][-2]
    return out


def metrics(run, p):
    e = run['x']-p.reference
    return {
        'RMSE_gap_mm': float(1e3*np.sqrt(np.mean(e*e))),
        'max_abs_error_mm': float(1e3*np.max(np.abs(e))),
        'min_gap_mm': float(1e3*np.min(run['x'])),
        'max_gap_mm': float(1e3*np.max(run['x'])),
        'max_current_A': float(np.max(run['i'])),
        'safety_violations': int(np.sum((run['x'] < p.gap_min) | (run['x'] > p.gap_max))),
    }


def make_plot(adaptive, nominal, p, outpath):
    fig, ax = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
    t = adaptive['t']
    ax[0].plot(t, 1e3*adaptive['x'], label='Adaptive neural–synergetic + CBF', lw=1.8)
    ax[0].plot(t, 1e3*nominal['x'], '--', label='Nominal synergetic', alpha=.8)
    ax[0].axhline(1e3*p.reference, color='k', lw=1, label='reference')
    ax[0].axhspan(1e3*p.gap_min, 1e3*p.gap_max, color='#2ca02c', alpha=.08, label='safe set')
    ax[0].set_ylim(1e3*(p.gap_min-0.001), 1e3*(p.gap_max+0.001))
    ax[0].text(5.35, 23.6, 'nominal response leaves safe set\nand is clipped from view',
               color='#ff7f0e', fontsize=8, ha='center', va='top')
    ax[0].set_ylabel('gap [mm]'); ax[0].legend(ncol=2, fontsize=8); ax[0].grid(alpha=.25)
    ax[1].plot(t, adaptive['i'], color='#d62728'); ax[1].axhline(p.current_max, ls=':', color='k')
    ax[1].set_ylabel('current [A]'); ax[1].grid(alpha=.25)
    ax[2].plot(t, adaptive['d'], label='unknown true thrust')
    ax[2].plot(t, adaptive['dhat'], label='neural estimate', alpha=.85)
    ax[2].set_ylabel('load [N]'); ax[2].legend(); ax[2].grid(alpha=.25)
    ax[3].plot(t, adaptive['psi'], color='#9467bd'); ax[3].set_ylabel(r'$\psi$ [m/s]')
    ax[3].set_xlabel('time [s]'); ax[3].grid(alpha=.25)
    fig.suptitle('Robust levitation under unmeasured axial thrust')
    fig.tight_layout(); fig.savefig(outpath, dpi=180); plt.close(fig)


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--output', default='results', help='output directory')
    ap.add_argument('--seed', type=int, default=7)
    # parse_known_args is intentional: Jupyter/IPython injects arguments such
    # as ``-f <kernel.json>`` that do not belong to this demonstration.
    args, _unknown = ap.parse_known_args(argv)
    outdir = Path(args.output); outdir.mkdir(parents=True, exist_ok=True)
    p = Params()
    pinn = RandomFeaturePINN(p, seed=args.seed); pinn.train(seed=args.seed+4)
    xs = np.linspace(p.gap_min, p.gap_max, 100)
    rel = np.abs(pinn.force(xs, 2.0)-true_force(xs, 2.0, p))/true_force(xs, 2.0, p)
    adaptive = simulate(AdaptiveSynergeticController(p, pinn, True, True), p)
    nominal = simulate(AdaptiveSynergeticController(p, pinn, False, False), p)
    report = {'PINN_mean_relative_error_percent': float(100*np.mean(rel)),
              'adaptive_safe': metrics(adaptive, p), 'nominal': metrics(nominal, p)}
    make_plot(adaptive, nominal, p, outdir/'levitation_demo.png')
    np.savez(outdir/'simulation_data.npz', **{f'adaptive_{k}':v for k,v in adaptive.items()},
             **{f'nominal_{k}':v for k,v in nominal.items()})
    (outdir/'metrics.json').write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
    print(f"Saved results to {outdir.resolve()}")


if __name__ == '__main__':
    main()


{
  "PINN_mean_relative_error_percent": 0.1642762372461559,
  "adaptive_safe": {
    "RMSE_gap_mm": 0.8899169820354368,
    "max_abs_error_mm": 4.499999999999999,
    "min_gap_mm": 13.724972540320751,
    "max_gap_mm": 18.5,
    "max_current_A": 3.756684840617847,
    "safety_violations": 0
  },
  "nominal": {
    "RMSE_gap_mm": 11385.448118464119,
    "max_abs_error_mm": 30678.897311651806,
    "min_gap_mm": 14.137624156387922,
    "max_gap_mm": 30692.897311651806,
    "max_current_A": 4.2,
    "safety_violations": 3671
  }
}
Saved results to C:\Users\User\results
